<a href="https://colab.research.google.com/github/Anshu-Tomar/CapstoneProject-Beyond-Predictive-Accuracy/blob/baseline-model/BaseLineModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!git clone https://github.com/Anshu-Tomar/CapstoneProject-Beyond-Predictive-Accuracy.git

fatal: destination path 'CapstoneProject-Beyond-Predictive-Accuracy' already exists and is not an empty directory.


In [5]:
%cd /content/CapstoneProject-Beyond-Predictive-Accuracy

/content/CapstoneProject-Beyond-Predictive-Accuracy


In [6]:
!git checkout baseline-model

Already on 'baseline-model'
Your branch is up to date with 'origin/baseline-model'.


###**Base Line Model**

##**Logistic Regression**
**RandomForest**
**XGBOOST**
**Support vector Machine**

In [7]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sb

df = pd.read_excel('/content/CapstoneProject-Beyond-Predictive-Accuracy/Diabetes Simple Diagnosis.csv.xlsx')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88380 entries, 0 to 88379
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   88380 non-null  int64  
 1   Age          88380 non-null  int64  
 2   Gender       88380 non-null  object 
 3   BMI          88380 non-null  int64  
 4   High_BP      88380 non-null  int64  
 5   FBS          88380 non-null  int64  
 6   HbA1c_level  88380 non-null  float64
 7   Smoking      88380 non-null  int64  
 8   Diagnosis    88380 non-null  int64  
dtypes: float64(1), int64(7), object(1)
memory usage: 6.1+ MB


In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from xgboost import XGBClassifier

In [9]:
!pip install xgboost

In [12]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, average_precision_score, brier_score_loss)

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
X = df[['Age', 'Gender', 'BMI', 'High_BP',
        'FBS', 'HbA1c_level', 'Smoking']]

y = df['Diagnosis']


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

numerical_features = [
    'Age', 'BMI', 'FBS', 'HbA1c_level'
]

categorical_features = [
    'Gender', 'High_BP', 'Smoking'
]
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])
print("Training records:", X_train.shape)
print("Testing records:", X_test.shape)

print("Training diagnosis:")
print(y_train.value_counts())

print("Testing diagnosis:")
print(y_test.value_counts())

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Training records: (70704, 7)
Testing records: (17676, 7)
Training diagnosis:
Diagnosis
0    63965
1     6739
Name: count, dtype: int64
Testing diagnosis:
Diagnosis
0    15991
1     1685
Name: count, dtype: int64
Processed training shape: (70704, 11)
Processed testing shape: (17676, 11)


In [25]:
"""max_iter=1000: the maximum number of learning rounds.
class_weight='balanced': only about 10% of patients are diabetic, so this makes a mistake on a diabetic patient cost more. This increases recall."""

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
numerical_features = [
    'Age', 'BMI', 'FBS', 'HbA1c_level'
]

categorical_features = [
    'Gender', 'High_BP', 'Smoking'
]

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

# model = LogisticRegression(max_iter=1000, class_weight='balanced')
# model.fit(X_train_processed, y_train)
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000
    ),

    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    'XGBoost': XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        random_state=42,
        eval_metric='logloss',
        n_jobs=-1
    ),
    'Decision Tree': DecisionTreeClassifier(
        random_state=42,
        class_weight=None
    ),

}



cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision'
}

results = []

for name, model in models.items():

    pipeline = Pipeline([
        ('preprocessing', preprocessor),
        ('classifier', model)
    ])

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    results.append({
        'Model': name,
        'Accuracy': scores['test_accuracy'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1': scores['test_f1'].mean(),
        'ROC_AUC': scores['test_roc_auc'].mean(),
        'PR_AUC': scores['test_pr_auc'].mean()
    })

baseline_results = pd.DataFrame(results)

baseline_results = baseline_results.sort_values(
    by='PR_AUC',
    ascending=False
)

baseline_results.round(4)
print(baseline_results.round(4))

print('Training done')

                 Model  Accuracy  Precision  Recall      F1  ROC_AUC  PR_AUC
2              XGBoost    0.9683     0.9889  0.6749  0.8022   0.9764  0.8839
1        Random Forest    0.9632     0.9053  0.6857  0.7803   0.9544  0.8454
0  Logistic Regression    0.9549     0.8760  0.6139  0.7217   0.9581  0.7950
3        Decision Tree    0.9436     0.6928  0.7329  0.7122   0.8512  0.5360
Training done


In [22]:
!pip install -q imbalanced-learn

In [24]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
print("Before SMOTE:")
print(y_train.value_counts())
print("SMOTE imported successfully")

Before SMOTE:
Diagnosis
0    63965
1     6739
Name: count, dtype: int64
SMOTE imported successfully
